In [1]:
# Optional: install required libraries for this lecture
%pip install -q openai pydantic


### Step 1: Enter your OpenRouter API key and initialize the client

This notebook uses **OpenRouter** through its OpenAI-compatible API.
You can enter your API key directly when the notebook runs. The key is hidden while you type.


In [2]:
import getpass
from openai import OpenAI
from pydantic import BaseModel, Field

OPENROUTER_API_KEY = getpass.getpass("Enter your OpenRouter API key: ")

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

print("OpenRouter client initialized:", bool(OPENROUTER_API_KEY))


Enter your OpenRouter API key: ··········
OpenRouter client initialized: True


### Step 2: Define a small schema

We'll capture a todo/calendar event with three fields.


In [3]:
class CalendarEvent(BaseModel):
    name: str = Field(description="the name")
    date: str = Field(description="the date")
    participants: list[str]

res = client.beta.chat.completions.parse(
    model="openai/gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Extract the event information."},
        {"role": "user", "content": "Alice and Bob are going to a science fair on Friday."},
    ],
    response_format=CalendarEvent,
)

print(res.choices[0].message.parsed.model_dump())


{'name': 'Science Fair', 'date': 'Friday', 'participants': ['Alice', 'Bob']}


### Alternative: Structured output via JSON mode

Ask for explicit keys and enable JSON mode to request a JSON object.


In [4]:
res2 = client.chat.completions.create(
    model="openai/gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": (
                "- Extract the event information.\n"
                "- Return a JSON object containing:\n"
                "    - the name of the event, using the key `event`\n"
                "    - the date of the event, using the key `date`\n"
                "    - an array of all participants, using the key `participants`\n"
            ),
        },
        {"role": "user", "content": "Alice and Bob are going to a science fair on Friday."},
    ],
    response_format={"type": "json_object"},
)

print(res2.choices[0].message.content)


{
    "event": "science fair",
    "date": "Friday",
    "participants": ["Alice", "Bob"]
}


### Try another OpenRouter model

OpenRouter provides access to many models through the same API. For example, you can replace the model above with another model available on OpenRouter.
